# 1. Information about the submission

## 1.1 Name and number of the assignment

Hallucination Detection in Tool Calling

## 1.2 Student name

Erik Shaikhiev

Tishchenko Margarita

Artemii Rubtcov

Roman Branovets

Andrej Mymrin

## 1.3 Codalab user ID / nickname / username

Not applicable for this local reproducibility notebook.

## 1.4 Additional comments

The notebook is written to be reproducible both inside the local repository and in a clean Colab-like environment. It uses the public GitHub repository `Eroouu/transformers_project.git` and the prepared ToolACE-derived datasets when they are available.

# 2. Technical Report

*Use Section 2 to describe results of your experiments as you would do writing a paper about your results. DO NOT insert code in this part. Only insert plots and tables summarizing results as needed. Use formulas if needed do described your methodology. The code is provided in Section 3.*

## 2.1 Methodology

The assignment asks us to detect span-level hallucinations in tool-calling dialogues. Following the provided task description, each example is represented in a RAGTruth-style schema: `query` is the user question, `context` is the tool output, `output` is the final model answer, and `hallucination_labels` contains character-level spans that mark unsupported text. The project builds on ToolACE dialogues and creates three corrupted subsets: contradiction with the tool output, overgeneration beyond the tool output, and missing-tool suggestions that require unavailable tools. Clean examples are kept as negative cases.

The repository implements the full pipeline in Python. Dataset construction and validation live in `data/`, while training and evaluation live in `src/`. The main improved model is a LettuceDetect-compatible token classifier fine-tuned on the generated span labels. During preprocessing, the tool output and user query are formatted as context/question, the final answer is tokenized as the second sequence, and only answer tokens receive binary labels: `supported` or `hallucination`. To handle class imbalance, the trainer can use inverse-frequency class weights.

For a reproducible run, the notebook first locates or clones `https://github.com/Eroouu/transformers_project.git`, installs the project dependencies, validates the final dataset split, trains the token classifier on `final_dataset_train`, and evaluates on `final_dataset_test`. The default configuration uses a short `max_steps` smoke training run so that the notebook can execute on limited hardware; setting `QUICK_RUN = False` switches to the full training schedule. Metrics are span-level Precision, Recall, and F1 computed by the repository evaluator through overlap between predicted and gold hallucination spans.

## 2.2 Discussion of results

The final dataset contains four files: `clean.jsonl`, `contradiction.jsonl`, `overgeneration.jsonl`, and `missing_tool.jsonl`. In the local prepared split, each corruption type has 2431 examples in total, with 1945 train and 486 test examples; the clean split has the same number of negative examples. This gives 7779 train examples and 1944 held-out test examples across all four files.

The heuristic `tool_overlap` baseline has very high recall but low precision because it flags many answer tokens that are simply absent from the raw tool output. On the local held-out test split it gives `TP=2099`, `FP=13582`, `FN=18`, `P=0.1339`, `R=0.9915`, `F1=0.2359`. The fine-tuned LettuceDetect-compatible model is the main improved method: the repository README records the latest local full-run result on the ToolACE test split as `P=0.9614`, `R=0.9719`, `F1=0.9666`, while LookBack Lens reached `P=0.4185`, `R=0.9756`, `F1=0.5858`. The code below recomputes the metrics for the current run and prints a comparison table.

# 3. Code

*Enter here all code used to produce your results submitted to Codalab. Add some comments and subsections to navigate though your solution.*

*In this part you are expected to develop yourself a solution of the task and provide a reproducible code:*
- *Using Python 3;*
- *Contains code for installation of all dependencies;*
- *Contains code for downloading of all the datasets used*;
- *Contains the code for reproducing your results (in other words, if a tester downloads your notebook she should be able to run cell-by-cell the code and obtain your experimental results as described in the methodology section)*.


*As a result, you code will be graded according to these criteria:*
- ***Readability**: your code should be well-structured preferably with indicated parts of your approach (Preprocessing, Model training, Evaluation, etc.).*
- ***Reproducibility**: your code should be reproduced without any mistakes with “Run all” mode (obtaining experimental part).*


## 3.1 Requirements

In [ ]:
# 3.1 Requirements and repository setup
# This cell works both inside the local repository and in a clean notebook runtime.

from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/Eroouu/transformers_project.git"
REPO_DIR = Path("transformers_project")

cwd = Path.cwd()
if (cwd / "src" / "train_lettucedetect.py").exists() and (cwd / "requirements.txt").exists():
    project_dir = cwd
else:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    project_dir = REPO_DIR.resolve()
    os.chdir(project_dir)

print(f"Project directory: {Path.cwd()}")
print(f"Repository URL: {REPO_URL}")

INSTALL_DEPS = True
if INSTALL_DEPS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)


## 3.2 Download the data

In [ ]:
# 3.2 Download / locate the data
# The project contains prepared final datasets. If they are not present, use README commands
# to build them from ToolACE and generated corruptions before running the training cell.

from pathlib import Path

DATASET_DIR = Path("final_dataset")
TRAIN_DIR = Path("final_dataset_train")
TEST_DIR = Path("final_dataset_test")
DATASET_FILES = ["clean.jsonl", "contradiction.jsonl", "overgeneration.jsonl", "missing_tool.jsonl"]

for dataset_dir in [DATASET_DIR, TRAIN_DIR, TEST_DIR]:
    missing = [name for name in DATASET_FILES if not (dataset_dir / name).exists()]
    if missing:
        raise FileNotFoundError(
            f"Missing files in {dataset_dir}: {missing}. "
            "Generate the dataset with scripts from data/ or pull the prepared artifacts."
        )

print("Found prepared datasets:")
for dataset_dir in [DATASET_DIR, TRAIN_DIR, TEST_DIR]:
    print(f"- {dataset_dir.resolve()}")


In [ ]:
# Quick look at one example in the required RAGTruth-style schema.

import json
from pprint import pprint

sample_path = TEST_DIR / "contradiction.jsonl"
with sample_path.open("r", encoding="utf-8") as f:
    sample = json.loads(next(f))

pprint({
    "query": sample.get("query", "")[:300],
    "context": sample.get("context", "")[:300],
    "output": sample.get("output", "")[:300],
    "hallucination_labels": sample.get("hallucination_labels", []),
    "corruption_type": sample.get("corruption_type"),
})


## 3.3 Preprocessing

In [ ]:
# 3.3 Preprocessing and validation
# Count examples and spans, then run the repository validator over the final dataset.

import json
import subprocess
import sys
from pathlib import Path

import pandas as pd


def summarize_dataset(dataset_dir: Path) -> pd.DataFrame:
    rows = []
    for name in DATASET_FILES:
        path = dataset_dir / name
        examples = 0
        examples_with_labels = 0
        spans = 0
        span_chars = 0
        with path.open("r", encoding="utf-8") as f:
            for line in f:
                item = json.loads(line)
                labels = item.get("hallucination_labels", [])
                examples += 1
                examples_with_labels += int(bool(labels))
                spans += len(labels)
                span_chars += sum(int(label["end"]) - int(label["start"]) for label in labels)
        rows.append({
            "split": dataset_dir.name,
            "file": name,
            "examples": examples,
            "examples_with_labels": examples_with_labels,
            "gold_spans": spans,
            "avg_span_chars": round(span_chars / spans, 2) if spans else 0.0,
        })
    return pd.DataFrame(rows)

stats = pd.concat(
    [summarize_dataset(DATASET_DIR), summarize_dataset(TRAIN_DIR), summarize_dataset(TEST_DIR)],
    ignore_index=True,
)
display(stats)

subprocess.run([sys.executable, "data/validate_corrupted_datasets.py", str(DATASET_DIR)], check=True)


## 3.4 My method of text processing

In [ ]:
# 3.4 Training, evaluation, and result table
# QUICK_RUN keeps the notebook practical for CPU/Colab smoke tests. Set QUICK_RUN = False
# for the full experiment used in the report.

import json
import os
import re
import subprocess
import sys
from pathlib import Path

import pandas as pd
import torch

QUICK_RUN = True
SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_NAME = "KRLabsOrg/lettucedect-base-modernbert-en-v1"
OUTPUT_DIR = Path("models/assignment_lettucedetect_quick" if QUICK_RUN else "models/assignment_lettucedetect_full")
MAX_STEPS = 20 if QUICK_RUN else -1
EPOCHS = 1 if QUICK_RUN else 3

train_cmd = [
    sys.executable,
    "src/train_lettucedetect.py",
    "--dataset", str(TRAIN_DIR),
    "--output_dir", str(OUTPUT_DIR),
    "--model", MODEL_NAME,
    "--device", DEVICE,
    "--epochs", str(EPOCHS),
    "--max_steps", str(MAX_STEPS),
    "--batch_size", "1",
    "--eval_batch_size", "2",
    "--gradient_accumulation_steps", "4",
    "--seed", str(SEED),
]
if DEVICE == "cuda":
    train_cmd.append("--fp16")

print("Training command:")
print(" ".join(map(str, train_cmd)))
subprocess.run(train_cmd, check=True)


def run_eval(method: str, **kwargs) -> dict:
    cmd = [
        sys.executable,
        "src/eval_baselines.py",
        "--dataset", str(TEST_DIR),
        "--method", method,
    ]
    if method == "lettucedetect":
        cmd += ["--lettuce_model", str(kwargs.get("lettuce_model", MODEL_NAME)), "--device", DEVICE]
    if method == "lookback_lens":
        cmd += ["--lookback_classifier", str(kwargs.get("lookback_classifier", "models/lookback_lens")), "--device", DEVICE]

    completed = subprocess.run(cmd, check=True, text=True, capture_output=True)
    print(completed.stdout)
    match = re.search(
        r"Method=(?P<method>\S+)\s+TP=(?P<tp>\d+) FP=(?P<fp>\d+) FN=(?P<fn>\d+) "
        r"P=(?P<precision>[0-9.]+) R=(?P<recall>[0-9.]+) F1=(?P<f1>[0-9.]+)",
        completed.stdout,
    )
    if not match:
        raise RuntimeError(f"Could not parse evaluator output:\n{completed.stdout}")
    row = match.groupdict()
    for key in ["tp", "fp", "fn"]:
        row[key] = int(row[key])
    for key in ["precision", "recall", "f1"]:
        row[key] = float(row[key])
    return row

results = []
results.append(run_eval("tool_overlap"))
results.append(run_eval("lettucedetect", lettuce_model=OUTPUT_DIR))

# Optional: evaluate LookBack Lens if a trained classifier is available.
LOOKBACK_DIR = Path("models/lookback_lens")
if LOOKBACK_DIR.exists():
    results.append(run_eval("lookback_lens", lookback_classifier=LOOKBACK_DIR))
else:
    print("Skipping LookBack Lens: models/lookback_lens was not found.")

results_df = pd.DataFrame(results)
display(results_df)

results_path = Path("outputs") / "assignment_metrics.json"
results_path.parent.mkdir(exist_ok=True)
results_path.write_text(json.dumps(results, indent=2), encoding="utf-8")
print(f"Saved metrics to {results_path.resolve()}")
